In [ ]:
# @title Build jitted functions, and possibly initialize random weights
import optax

def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # Deeper one-step predictor.
  predictor = graphcast.GraphCast(model_config, task_config)

  # Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to
  # from/to float32 to/from BFloat16.
  predictor = casting.Bfloat16Cast(predictor)

  # Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from
  # BFloat16 happens after applying normalization to the inputs/targets.
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # Wraps everything so the one-step model can produce trajectories.
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor


@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


@hk.transform_with_state
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))





def grads_fn(params, state, inputs, targets, forcings, model_config, task_config):
    def _aux(params, state, i, t, f):
        (loss, diagnostics), next_state = loss_fn.apply(params, state, jax.random.PRNGKey(0), model_config, task_config, i, t, f)
        return loss, (diagnostics, next_state)
    (loss, (diagnostics, next_state)), grads = jax.value_and_grad(_aux, has_aux=True)(params, state, inputs, targets, forcings)
    return loss, diagnostics, next_state, grads




# Jax doesn't seem to like passing configs as args through the jit. Passing it
# in via partial (instead of capture by closure) forces jax to invalidate the
# jit cache if you change configs.
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# Always pass params and state, so the usage below are simpler
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# Our models aren't stateful, so the state is always empty, so just return the
# predictions. This is requiredy by our rollout code, and generally simpler.
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]




init_jitted = jax.jit(with_configs(run_forward.init))

if params is None:
  params, state = init_jitted(
      rng=jax.random.PRNGKey(0),
      inputs=train_inputs,
      targets_template=train_targets,
      forcings=train_forcings)

loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))



# grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))


run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
    run_forward.apply))))

# Run the model

Note that the cell below may take a while (possibly minutes) to run the first time you execute them, because this will include the time it takes for the code to compile. The second time running will be significantly faster.

This use the python loop to iterate over prediction steps, where the 1-step prediction is jitted. This has lower memory requirements than the training steps below, and should enable making prediction with the small GraphCast model on 1 deg resolution data for 4 steps.

In [ ]:
# @title Autoregressive rollout (loop in python)

assert model_config.resolution in (0, 360. / eval_inputs.sizes["lon"]), (
  "Model resolution doesn't match the data resolution. You likely want to "
  "re-filter the dataset list, and download the correct data.")

print("Inputs:  ", eval_inputs.dims.mapping)
print("Targets: ", eval_targets.dims.mapping)
print("Forcings:", eval_forcings.dims.mapping)

predictions = rollout.chunked_prediction(
    run_forward_jitted,
    rng=jax.random.PRNGKey(0),
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings)
predictions

Inputs:   {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Targets:  {'batch': 1, 'time': 6, 'lat': 181, 'lon': 360, 'level': 13}
Forcings: {'batch': 1, 'time': 6, 'lat': 181, 'lon': 360}


/usr/local/lib/python3.11/dist-packages/graphcast/rollout.py:128: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_target_steps = targets_template.dims["time"]
/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:202: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  scan_length = targets_template.dims['time']
/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:115: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension

<xarray.Dataset> Size: 130MB
Dimensions:                  (time: 6, batch: 1, lat: 181, lon: 360, level: 13)
Coordinates:
  * lon                      (lon) float32 1kB 0.0 1.0 2.0 ... 357.0 358.0 359.0
  * lat                      (lat) float32 724B -90.0 -89.0 -88.0 ... 89.0 90.0
  * level                    (level) int32 52B 50 100 150 200 ... 850 925 1000
  * time                     (time) timedelta64[ns] 48B 06:00:00 ... 1 days 1...
Dimensions without coordinates: batch
Data variables:
    10m_u_component_of_wind  (time, batch, lat, lon) float32 2MB -0.3918 ... ...
    10m_v_component_of_wind  (time, batch, lat, lon) float32 2MB -0.3055 ... ...
    2m_temperature           (time, batch, lat, lon) float32 2MB 247.8 ... 248.2
    geopotential             (time, batch, level, lat, lon) float32 20MB 1.99...
    mean_sea_level_pressure  (time, batch, lat, lon) float32 2MB 9.949e+04 .....
    specific_humidity        (time, batch, level, lat, lon) float32 20MB 2.89...
    temperature              (time, batch, level, lat, lon) float32 20MB 239....
    total_precipitation_6hr  (time, batch, lat, lon) float32 2MB 5.939e-05 .....
    u_component_of_wind      (time, batch, level, lat, lon) float32 20MB 1.67...
    v_component_of_wind      (time, batch, level, lat, lon) float32 20MB 0.13...
    vertical_velocity        (time, batch, level, lat, lon) float32 20MB -0.0...

In [ ]:
# @title Choose predictions to plot

plot_pred_variable = widgets.Dropdown(
    options=predictions.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_pred_level = widgets.Dropdown(
    options=predictions.coords["level"].values,
    value=500,
    description="Level")
plot_pred_robust = widgets.Checkbox(value=True, description="Robust")
plot_pred_max_steps = widgets.IntSlider(
    min=1,
    max=predictions.dims["time"],
    value=predictions.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_pred_variable,
    plot_pred_level,
    plot_pred_robust,
    plot_pred_max_steps,
    widgets.Label(value="Run the next cell to plot the predictions. Rerunning this cell clears your selection.")
])

<ipython-input-20-478ca7e692f5>:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  max=predictions.dims["time"],
<ipython-input-20-478ca7e692f5>:15: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  value=predictions.dims["time"],


In [ ]:
# @title Plot predictions

plot_size = 5
plot_max_steps = min(predictions.dims["time"], plot_pred_max_steps.value)

data = {
    "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Predictions": scale(select(predictions, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps) -
                        select(predictions, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}
fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

# plot_data(data, fig_title, plot_size, plot_pred_robust.value)


<ipython-input-21-aa34a470f71e>:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  plot_max_steps = min(predictions.dims["time"], plot_pred_max_steps.value)


# Train the model

The following operations require a large amount of memory and, depending on the accelerator being used, will only fit the very small "random" model on low resolution data. It uses the number of training steps selected above.

The first time executing the cell takes more time, as it include the time to jit the function.

In [ ]:
# @title Loss computation (autoregressive loss over multiple steps)
loss, diagnostics = loss_fn_jitted(
    rng=jax.random.PRNGKey(0),
    inputs=train_inputs,
    targets=train_targets,
    forcings=train_forcings)
print("Loss:", float(loss))

/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:292: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  scan_length = targets.dims['time']
/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:115: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_inputs = inputs.dims['time']


In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
In normalization.py loss_and_predictions
In casting.py loss_and_predictions
In graphcast.py loss_and_predictions
In losses.py weighted_mse_per_level
Loss: 1.685546875


In [ ]:
# remove `with_params` from jitted grads function
grads_fn_jitted = jax.jit(with_configs(grads_fn))

# setup optimiser
lr = 1e-3
optimiser = optax.adam(lr, b1=0.9, b2=0.999, eps=1e-8)
old_params = params
opt_state = optimiser.init(old_params)

# calculate loss and gradients
loss, diagnostics, next_state, grads = grads_fn_jitted(old_params, state, train_inputs, train_targets, train_forcings)

# update
updates, opt_state = optimiser.update(grads, opt_state)
new_params = optax.apply_updates(old_params, updates)

# # @title Gradient computation (backprop through time)
# loss, diagnostics, next_state, grads = grads_fn_jitted(
#     inputs=train_inputs,
#     targets=train_targets,
#     forcings=train_forcings)
# mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])
# print(f"Loss: {loss:.4f}, Mean |grad|: {mean_grad:.6f}")

/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:292: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  scan_length = targets.dims['time']
/usr/local/lib/python3.11/dist-packages/graphcast/autoregressive.py:115: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  num_inputs = inputs.dims['time']


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 26846808568 bytes.

In [ ]:
import jax.numpy as jnp
import os
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

def flatten_dict(d, parent_key='', sep='//'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def save_model_params(d, file_path):
    flat_dict = flatten_dict(d)
    # Convert JAX arrays to NumPy for saving
    np_dict = {k: np.array(v) if isinstance(v, jnp.ndarray) else v for k, v in flat_dict.items()}
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    np.savez(file_path, **np_dict)

def unflatten_dict(d, sep='//'):
    result_dict = {}
    for flat_key, value in d.items():
        keys = flat_key.split(sep)
        d_nested = result_dict
        for key in keys[:-1]:
            if key not in d_nested:
                d_nested[key] = {}
            d_nested = d_nested[key]
        d_nested[keys[-1]] = value
    return result_dict

def load_model_params(file_path):
    with np.load(file_path, allow_pickle=True) as npz_file:
        # Convert NumPy arrays back to JAX arrays
        jax_dict = {k: jnp.array(v) for k, v in npz_file.items()}
    return unflatten_dict(jax_dict)

params_dir = '/content/drive/MyDrive/'

os.makedirs(params_dir, exist_ok=True)


# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d_%H-%M")

# Create the file name with the current date
file_name = f"params_{current_date}.npz"

# Join the directory and file name
params_path = os.path.join(params_dir, file_name)

save_model_params(params, params_path)

# Later, load the model parameters
params = load_model_params(params_path)

In [ ]:

# Define the function to run the model
def run_model(params, state, inputs, targets_template, forcings):
    predictions = run_forward_jitted(
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets_template,
        forcings=forcings,
        params=old_params,
        state=state
    )
    return predictions

# Prepare the targets template (filled with NaNs)
targets_template = train_targets * np.nan

# Run the model with the fine-tuned parameters
predictions = run_model(old_params, state, train_inputs, targets_template, train_forcings)

# Extract the variable of interest (e.g., temperature)
temperature_pred = predictions['2m_temperature']

# Plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Select a time step to visualize
time_step = 1

# Ensure the DataArray is 2-dimensional (latitude, longitude)
temperature_2d = temperature_pred.isel(time=time_step).squeeze()

# Plotting
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([68, 98, 6, 38])  # India focus

# Plot the predicted temperature
temperature_2d.plot.pcolormesh(
    ax=ax, transform=ccrs.PlateCarree(), cmap='coolwarm',
    cbar_kwargs={'label': 'Temperature (K)'})

ax.coastlines()
ax.set_title(f"Predicted 2m Air Temperature over India at Time Step {time_step}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Function to run the model and get predictions
def run_model(params, state, inputs, targets_template, forcings):
    predictions = run_forward_jitted(
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets_template,
        forcings=forcings,
        params=params,
        state=state
    )
    return predictions

# Prepare the targets template (filled with NaNs)
targets_template = eval_targets * np.nan

# Run the model with the old (unfinetuned) parameters
predictions_old = run_model(old_params, state, eval_inputs, targets_template, eval_forcings)

# Run the model with the finetuned parameters
predictions_finetuned = run_model(new_params, state, eval_inputs, targets_template, eval_forcings)

# Extract the variable of interest (e.g., '2m_temperature')
variable_name = '2m_temperature'
temperature_pred_old = predictions_old[variable_name]
temperature_pred_finetuned = predictions_finetuned[variable_name]
temperature_targets = eval_targets[variable_name]

# Define the time steps to visualize
time_steps = [0, 1]  # Adjust as needed

# Function to plot predictions over India
def plot_predictions(time_step):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Old predictions
    ax = axs[0]
    temp_old = temperature_pred_old.isel(time=time_step).squeeze()
    temp_old.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='coolwarm',
        cbar_kwargs={'label': 'Temperature (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Old Predictions at Time Step {time_step}")

    # Finetuned predictions
    ax = axs[1]
    temp_finetuned = temperature_pred_finetuned.isel(time=time_step).squeeze()
    temp_finetuned.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='coolwarm',
        cbar_kwargs={'label': 'Temperature (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Finetuned Predictions at Time Step {time_step}")

    # Difference between finetuned and old predictions
    ax = axs[2]
    difference = temp_finetuned - temp_old
    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Difference at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

# Plot predictions for each time step
for time_step in time_steps:
    plot_predictions(time_step)

# Function to plot difference between finetuned predictions and ground truth
def plot_difference_with_targets(time_step):
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Difference between finetuned predictions and evaluation targets
    temp_finetuned = temperature_pred_finetuned.isel(time=time_step).squeeze()
    temp_targets = temperature_targets.isel(time=time_step).squeeze()
    difference = temp_finetuned - temp_targets

    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Finetuned Prediction vs Ground Truth at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

    return (difference)

# Function to plot difference between base predictions and ground truth
def plot_difference_with_targets_base(time_step):
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Difference between finetuned predictions and evaluation targets
    temp_old = temperature_pred_old.isel(time=time_step).squeeze()
    temp_targets = temperature_targets.isel(time=time_step).squeeze()
    difference = temp_old - temp_targets

    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Base Prediction vs Ground Truth at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

    return (difference)

# Plot differences with ground truth for each time step
differences_finetuned = []
differences_base = []
for time_step in time_steps:
    differences_finetuned.append(plot_difference_with_targets(time_step))
    differences_base.append(plot_difference_with_targets_base(time_step))

In [ ]:
# @title Plot predictions

plot_size = 5
plot_max_steps = min(predictions_finetuned.dims["time"], plot_pred_max_steps.value)

data = {
    "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Predictions": scale(select(predictions_finetuned, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps) -
                        select(predictions_finetuned, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}
fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

# plot_data(data, fig_title, plot_size, plot_pred_robust.value)


In [ ]:
# @title Plot predictions

plot_size = 5
plot_max_steps = min(predictions_old.dims["time"], plot_pred_max_steps.value)

data = {
    "Targets": scale(select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Predictions": scale(select(predictions_old, plot_pred_variable.value, plot_pred_level.value, plot_max_steps), robust=plot_pred_robust.value),
    "Diff": scale((select(eval_targets, plot_pred_variable.value, plot_pred_level.value, plot_max_steps) -
                        select(predictions_old, plot_pred_variable.value, plot_pred_level.value, plot_max_steps)),
                       robust=plot_pred_robust.value, center=0),
}
fig_title = plot_pred_variable.value
if "level" in predictions[plot_pred_variable.value].coords:
  fig_title += f" at {plot_pred_level.value} hPa"

# plot_data(data, fig_title, plot_size, plot_pred_robust.value)


In [ ]:

params_dir = '/content/drive/MyDrive/'

os.makedirs(params_dir, exist_ok=True)

file_name_base = f"base_model_params_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.npz"
params_path_base = os.path.join(params_dir, file_name_base)
save_model_params(old_params, params_path_base)

# 2. Save the fine-tuned model parameters:
file_name_finetuned = f"finetuned_model_params_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.npz"
params_path_finetuned = os.path.join(params_dir, file_name_finetuned)
save_model_params(new_params, params_path_finetuned)

In [ ]:
def compute_mse(predictions, targets):
    return ((predictions - targets) ** 2).mean()

# Compute the MSE for each time step and aggregate
mse_finetuned = []
mse_base = []

for time_step in time_steps:
    temp_finetuned = temperature_pred_finetuned.isel(time=time_step).squeeze()
    temp_old = temperature_pred_old.isel(time=time_step).squeeze()
    temp_targets = temperature_targets.isel(time=time_step).squeeze()

    mse_finetuned.append(compute_mse(temp_finetuned, temp_targets))
    mse_base.append(compute_mse(temp_old, temp_targets))

# Compute the overall loss by averaging the MSE over all time steps
overall_loss_finetuned = np.mean(mse_finetuned)
overall_loss_base = np.mean(mse_base)

print(f"Overall Loss (Finetuned Predictions): {overall_loss_finetuned}")
print(f"Overall Loss (Base Predictions): {overall_loss_base}")

In [ ]:
type(differences_finetuned[0])

In [ ]:
loss, diagnostics = loss_fn_jitted(
    rng=jax.random.PRNGKey(0),
    inputs=train_inputs,
    targets=train_targets,
    forcings=train_forcings)
print("Loss:", float(loss))

In [ ]:
# @title Autoregressive rollout (keep the loop in JAX)
print("Inputs:  ", train_inputs.dims.mapping)
print("Targets: ", train_targets.dims.mapping)
print("Forcings:", train_forcings.dims.mapping)

predictions = run_forward_jitted(
    rng=jax.random.PRNGKey(0),
    inputs=train_inputs,
    targets_template=train_targets * np.nan,
    forcings=train_forcings)
predictions

In [ ]:
train_inputs